# Patches — Bloc 3a, Sessió 3
### Captura, playback i efectes offline (model blocking)

**Com obrir aquest notebook:** Fes clic a l'enllaç del Classroom. Colab el carregarà directament des de GitHub. Per guardar els teus canvis: `File → Save a copy in Drive`.

Aquest notebook és per a la **demo guiada en directe**.

In [ ]:
import numpy as np
import soundfile as sf
import librosa
import matplotlib.pyplot as plt
from IPython.display import Audio

sample_rate = 44100

## 1. Gravar amb el micròfon (model blocking a Colab)

A Colab no tenim `sounddevice` directament, però podem gravar amb JavaScript del navegador. El resultat és el mateix: un array NumPy.

In [ ]:
# Gravació a Colab via navegador
from google.colab import output
from base64 import b64decode
from IPython.display import Javascript, display

RECORD_JS = """
const sleep = t => new Promise(r => setTimeout(r, t))
const b2text = blob => new Promise(r => {
  const reader = new FileReader()
  reader.onloadend = e => r(e.srcElement.result)
  reader.readAsDataURL(blob)
})
async function record(seconds) {
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  const recorder = new MediaRecorder(stream)
  const chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(seconds * 1000)
  recorder.stop()
  await new Promise(r => recorder.onstop = r)
  stream.getTracks().forEach(t => t.stop())
  return await b2text(new Blob(chunks))
}
"""

def record_audio(seconds=3):
    display(Javascript(RECORD_JS))
    s = output.eval_js(f'record({seconds})')
    b = b64decode(s.split(',')[1])
    with open('gravacio_raw.webm', 'wb') as f:
        f.write(b)

print("Gravant 3 segons... fes un so ara!")
record_audio(seconds=3)
!ffmpeg -y -i gravacio_raw.webm -ar 44100 -ac 1 gravacio.wav -loglevel quiet
print("Gravat com gravacio.wav")

In [ ]:
# Carregar la gravació com a array
data, sr = librosa.load('gravacio.wav', sr=sample_rate)

print("Shape:", data.shape)
print("Durada:", len(data)/sample_rate, "s")

librosa.display.waveshow(data, sr=sr)
plt.title("La teva gravació")
plt.show()

Audio(data, rate=sample_rate)

## 2. Les funcions d'efectes

Totes reben un array `data`, retornen un array processat.

In [ ]:
# --- EFECTES --- #

def reverse(data):
    return data[::-1]

def fade_in(data, duration, sample_rate=44100):
    n = min(int(duration * sample_rate), len(data))
    env = np.ones(len(data))
    env[:n] = np.linspace(0, 1, n)
    return data * env

def fade_out(data, duration, sample_rate=44100):
    n = min(int(duration * sample_rate), len(data))
    env = np.ones(len(data))
    env[-n:] = np.linspace(1, 0, n)
    return data * env

def echo(data, delay_seconds, decay=0.5, sample_rate=44100):
    delay_samples = int(delay_seconds * sample_rate)
    result = np.copy(data)  # IMPORTANT: copia, no referencia!
    result[delay_samples:] += data[:-delay_samples] * decay
    return result

def delay_multi(data, delay_seconds, decay=0.5, n_repeats=4, sample_rate=44100):
    result = np.copy(data)
    delay_samples = int(delay_seconds * sample_rate)
    for i in range(1, n_repeats + 1):
        start = i * delay_samples
        if start >= len(data):
            break
        result[start:] += data[:len(data)-start] * (decay ** i)
    return result

def distortion(data, drive=5.0, threshold=0.7):
    clipped = np.clip(data * drive, -threshold, threshold)
    return clipped / np.max(np.abs(clipped))

def tremolo(data, rate=5.0, depth=0.5, sample_rate=44100):
    t = np.linspace(0, len(data)/sample_rate, len(data), endpoint=False)
    lfo = 1 - depth * (0.5 + 0.5 * np.sin(2 * np.pi * rate * t))
    return data * lfo

def ring_modulation(data, carrier_freq=200.0, sample_rate=44100):
    t = np.linspace(0, len(data)/sample_rate, len(data), endpoint=False)
    carrier = np.sin(2 * np.pi * carrier_freq * t)
    return data * carrier

def playback_speed(data, factor, sample_rate=44100):
    """Canvia velocitat de reproduccio (i to i durada). NO es pitch shift real."""
    new_sr = int(sample_rate * factor)
    return data, new_sr

print("Efectes carregats!")

## 3. DEMO — provar cada efecte

🎤 *Canvia els paràmetres i escolta el resultat. Quins paràmetres el fan sonar millor/pitjor/més interessant?*

In [ ]:
# Reverse
print("Reverse:")
Audio(reverse(data), rate=sample_rate)

In [ ]:
# Echo — prova decay=0.3 vs 0.7, delay=0.1 vs 0.5
print("Echo:")
Audio(echo(data, delay_seconds=0.2, decay=0.5), rate=sample_rate)

In [ ]:
# Distorsio — prova drive=2 vs 10
print("Distorsio:")
Audio(distortion(data, drive=5.0, threshold=0.7), rate=sample_rate)

In [ ]:
# Tremolo — prova rate=2 vs 10, depth=0.3 vs 0.9
print("Tremolo:")
Audio(tremolo(data, rate=5.0, depth=0.7), rate=sample_rate)

In [ ]:
# Ring modulation — prova carrier_freq=80 vs 500
print("Ring modulation:")
Audio(ring_modulation(data, carrier_freq=200.0), rate=sample_rate)

In [ ]:
# Playback speed — prova factor=0.5 (mes lent/greu) vs 2.0 (mes rapid/agut)
# Per que canvia el to? (Pensa en que representa sample_rate de la Sessio 1)
print("Playback speed x0.5:")
data_slow, sr_slow = playback_speed(data, factor=0.5)
display(Audio(data_slow, rate=sr_slow))

print("Playback speed x2.0:")
data_fast, sr_fast = playback_speed(data, factor=2.0)
display(Audio(data_fast, rate=sr_fast))

## 4. DEMO — encadenar efectes

🎤 *Ara combina efectes. Quin ordre dóna resultats més interessants?*

In [ ]:
# Exemple: distorsio + echo + fade out
cadena = data
cadena = distortion(cadena, drive=4.0)
cadena = echo(cadena, delay_seconds=0.15, decay=0.4)
cadena = fade_out(cadena, duration=0.5)

librosa.display.waveshow(cadena, sr=sample_rate)
plt.title("distortion → echo → fade_out")
plt.show()

Audio(cadena, rate=sample_rate)

## 5. DEMO — l'error de referència vs. còpia

🎤 *Per què cal `np.copy`? Veiem-ho en directe.*

In [ ]:
original = np.array([1.0, 2.0, 3.0, 4.0, 5.0])

# MAL: referencia
result_mal = original
result_mal[0] = 999
print("original despres de modificar result_mal:", original)  # original HA CANVIAT!

original = np.array([1.0, 2.0, 3.0, 4.0, 5.0])  # reset

# BE: copia
result_be = np.copy(original)
result_be[0] = 999
print("original despres de modificar result_be:", original)  # original NO ha canviat